# Apply the synthetic Cost Center allocation policy
Manual maintenance on loaded DEV, then PROD. Update the Git folder and pause ingestion jobs and dashboard queries. This notebook runs on Spark compute (classic or serverless), not a SQL Warehouse.
Existing source cost centers take priority. Missing centers in West Europe, North Europe, France Central, Sweden Central and UK South become CostCenter_Europe. Global and explicitly configured headquarters regions become CostCenter_Corporate. Other cases remain Unallocated Costs. No additional Corporate regions are configured initially.
Only Gold v_cost_allocation and dm_cost_by_scope_service_month are published. Source Parquet, Bronze, Silver, Gold facts, dim_billing_scope and security assignments remain unchanged. The notebook checks loaded months, row counts and monetary totals before publishing, then reconciles the result. Do not resume jobs after a failed check; correct the cause and rerun.
After DEV and PROD pass, execute updated security SQL 04 then 05 in PROD and redeploy the dashboard. See platform/common/sql/gold/README.md.

In [ ]:
from pathlib import Path
import sys

source_root = next((root / 'src' for root in (Path.cwd(), *Path.cwd().parents) if (root / 'src').is_dir()), None)
if source_root is None:
    raise ValueError('Open this notebook from the updated project Git folder.')
if str(source_root) not in sys.path:
    sys.path.insert(0, str(source_root))

In [ ]:
ENVIRONMENT = 'dev'
CONFIRMATION = ''  # Set to APPLY_COST_CENTER_ALLOCATION after reading the instructions.

if ENVIRONMENT not in {'dev', 'prod'}:
    raise ValueError('environment must be dev or prod')
if CONFIRMATION != 'APPLY_COST_CENTER_ALLOCATION':
    raise ValueError('Set CONFIRMATION = APPLY_COST_CENTER_ALLOCATION.')

In [ ]:
from finops_cloud.config import load_config
from finops_cloud.medallion.cost_allocation import apply_cost_allocation

config = load_config(ENVIRONMENT)
print(apply_cost_allocation(spark, config, CONFIRMATION))
allocation_view = config.table('cost_allocation_view', 'gold')
display(spark.sql(f'''
    SELECT billing_month, cost_center_allocated AS cost_center,
           allocation_method, count(*) AS source_rows, SUM(BilledCost) AS billed_cost,
           100 * SUM(BilledCost) / nullif(SUM(SUM(BilledCost)) OVER (PARTITION BY billing_month), 0)
             AS share_of_month_percent
    FROM {allocation_view}
    GROUP BY billing_month, cost_center_allocated, allocation_method
    ORDER BY billing_month, billed_cost DESC
'''))